# Lab 5: Introduction to GPU Computing with Python

**Duration:** 3 hours. **Level:** beginner. **Work mode:** pairs or individual.

| Student name | Station no. | Date |
|---|---|---|
| | | |

## 1. Overview
A CPU does a few things very fast, one after another. A GPU does thousands of small things at the same time. This is why the GPU of the Jetson runs AI models. In this lab you check that CUDA works, use the GPU from Python with two existing libraries (**CuPy**, which copies the NumPy style, and **PyTorch**), find out **where the GPU wins and where it loses**, and time the same calculation as in Lab 4 on the GPU. At the end you build your first project, the **Jetson System Monitor**: a small dashboard that shows the memory, the CPU and GPU activity, the temperatures and the time of your Jetson, and you watch the GPU load rise when you give it work.

**By the end of this lab you can:**
1. Check that CUDA, CuPy and PyTorch use the GPU of the Jetson.
2. Copy data to the GPU and back, and explain why the copy can cost more than the calculation.
3. Show with measurements where the GPU is faster than the CPU and where it is not.
4. Time a matrix multiplication on the CPU, with CuPy and with PyTorch, and compare the power modes.
5. Build a system monitor that reads memory, CPU load, GPU load and temperatures, and logs them.

**How this notebook works.** Run a cell with **Shift + Enter**, from the top. Type your answers in the cells that say *Your answers*. Pictures and live video appear under the cell that makes them. Press **Ctrl + S** often.

> **Scope.** You use existing libraries. You do not write CUDA kernels in this lab; the libraries do the work on the GPU for you.

Run each cell with **Shift + Enter**. Read the code first, then run it, then write your results in the answer cells.

A line that starts with `!` is a Linux command and `%%bash` makes a cell a small Linux session. The output appears under the cell. For programs that fill the screen (`jtop`), open a terminal inside JupyterLab: **File, New, Terminal**, and drag its tab beside this notebook.

---

## Part A: Check CUDA on the Jetson (15 min)

### Step 1: Your Workspace
This notebook lives in the `lab5` folder inside your own folder. Everything this lab makes is saved here.

In [ ]:
%%bash
pwd
ls

### Step 2: Look for the CUDA Compiler
`grep` keeps only the lines with the word `cuda`. `nvcc` is the CUDA compiler.

In [ ]:
%%bash
ls /usr/local/ | grep cuda
/usr/local/cuda/bin/nvcc --version

If `nvcc` is not found by its short name, CUDA is installed but not in your `PATH`. The cell above uses the full path, which always works. To make `nvcc` work in every new terminal, add it to your `.bashrc` once:

```
echo 'export PATH=/usr/local/cuda/bin:$PATH' >> ~/.bashrc
```

| Item | Your answer |
|---|---|
| CUDA folders in `/usr/local` | |
| CUDA version in the last line of `nvcc --version` | |

### Step 3: Watch the GPU in jtop
Open a **Terminal tab** (File, New, Terminal) and run `jtop`. Press `2` for the GPU page. Note the GPU load now (it should be near 0%). You watch it change in Parts C and E.

> **Note.** The Orin Nano GPU has 1024 CUDA cores and uses a power mode you set in Lab 1. In MAXN SUPER the GPU runs faster than in 15W. Always write down the mode when you record a timing.

## Part B: Check CuPy and Read the GPU (20 min)

### Step 1: Check the Installed Packages
You install nothing in this lab. The Lab 0 setup already installed NumPy, CuPy and PyTorch. You should see three lines. The NumPy version must start with 1 and the CuPy line should end with a 13 version. If a line is missing, call your instructor.

In [ ]:
!pip list 2>/dev/null | grep -i -E "^(numpy|cupy-cuda12x|torch) " 

## Part B, Step 2: Check That CuPy Works
You should see a version number and `[0 2 4 6 8]`. The first run can take several seconds.

In [ ]:
import cupy as cp
print(cp.__version__)
print(cp.arange(5) * 2)

## Part B, Step 3: Read the GPU Properties

In [ ]:
import cupy as cp

props = cp.cuda.runtime.getDeviceProperties(0)
name = props["name"]
if isinstance(name, bytes):
    name = name.decode()

print("CuPy version:", cp.__version__)
print("GPU count:", cp.cuda.runtime.getDeviceCount())
print("Name:", name)
print("Compute capability:", props["major"], props["minor"])
print("Multiprocessors (SMs):", props["multiProcessorCount"])
print("Max threads per block:", props["maxThreadsPerBlock"])
print("Warp size:", props["warpSize"])
free, total = cp.cuda.runtime.memGetInfo()
print(f"Free / total memory: {free / 1024**3:.2f} / {total / 1024**3:.2f} GB")

## Part C, Step 1: Same Code, Different Library
The two results look the same, but the second was calculated on the GPU. Run the last line to see the type.

In [ ]:
import numpy as np
import cupy as cp
a = np.arange(10)
g = cp.arange(10)
print(a * 2)
print(g * 2)
type(g)
g.shape, g.dtype

## Part C, Step 2: Copy Data to the GPU and Back
Run this cell **twice** and record the second run.

In [ ]:
import time
import numpy as np
import cupy as cp

n = 4096
a_cpu = np.random.rand(n, n).astype(np.float32)

# Warm-up: the first GPU call loads libraries
_ = cp.asarray(a_cpu)
cp.cuda.Device().synchronize()

start = time.perf_counter()
a_gpu = cp.asarray(a_cpu)
cp.cuda.Device().synchronize()
copy_ms = (time.perf_counter() - start) * 1000

start = time.perf_counter()
b_gpu = a_gpu * 2.0 + 1.0
cp.cuda.Device().synchronize()
calc_ms = (time.perf_counter() - start) * 1000

start = time.perf_counter()
b_cpu = cp.asnumpy(b_gpu)
back_ms = (time.perf_counter() - start) * 1000

print(f"Copy to GPU : {copy_ms:.2f} ms")
print(f"Calculation : {calc_ms:.2f} ms")
print(f"Copy back   : {back_ms:.2f} ms")

## Part C, Step 3: Where the GPU Wins and Loses
This times the same calculation for four array sizes on the CPU and on the GPU.

In [ ]:
import time
import numpy as np
import cupy as cp


def time_cpu(x, repeats):
    start = time.perf_counter()
    for _ in range(repeats):
        y = x * 2.0 + 1.0
    return (time.perf_counter() - start) / repeats * 1000


def time_gpu(x, repeats):
    y = x * 2.0 + 1.0  # warm-up
    cp.cuda.Device().synchronize()
    start = time.perf_counter()
    for _ in range(repeats):
        y = x * 2.0 + 1.0
    cp.cuda.Device().synchronize()
    return (time.perf_counter() - start) / repeats * 1000


for n in [100, 10_000, 1_000_000, 10_000_000]:
    x_cpu = np.random.rand(n).astype(np.float32)
    x_gpu = cp.asarray(x_cpu)
    cpu_ms = time_cpu(x_cpu, 20)
    gpu_ms = time_gpu(x_gpu, 20)
    print(f"{n:>10,} items   CPU {cpu_ms:8.3f} ms   GPU {gpu_ms:8.3f} ms")

## Part D, Step 1: Check That PyTorch Sees the GPU
If `CUDA available` is `False`, or the version ends with `+cpu`, do **not** run `pip install torch`. Call your instructor.

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    print("Compute capability:", torch.cuda.get_device_capability(0))

## Part D, Step 2: Tensors on the CPU and the GPU
`x.device` is `cpu` and `y.device` is `cuda:0`. `.cpu()` copies a tensor back.

In [ ]:
import torch
x = torch.rand(3, 3)
print(x.device)
y = x.to("cuda")
print(y.device)
z = torch.rand(3, 3, device="cuda")
print(y + z)
print((y + z).cpu())

## Part E, Step 1: Check the Power Mode
Now repeat the test from Lab 4 on the GPU. Use the same matrix sizes and the same power mode as your baseline. The mode must match the mode in your Lab 4 table. Some commands start with `sudo` (run as the administrator). A notebook has no terminal where `sudo` can ask for your password, so run the next cell once and type your password in the box (nothing is shown).


In [ ]:
import labview
labview.sudo_login()

In [ ]:
!sudo nvpmodel -q

Power mode now:

## Part E, Step 2: Time the Matrix Multiplication with CuPy
Run the next cell **twice** and record the second run. Open `jtop` in a Terminal tab to watch the GPU load.

In [ ]:
import time
import cupy as cp

sizes = [256, 512, 1024, 2048]

# Warm-up: the first GPU call loads libraries and prepares the GPU
w = cp.random.rand(64, 64, dtype=cp.float32)
_ = w @ w
cp.cuda.Device().synchronize()

for n in sizes:
    a = cp.random.rand(n, n, dtype=cp.float32)
    b = cp.random.rand(n, n, dtype=cp.float32)
    cp.cuda.Device().synchronize()

    times = []
    for _ in range(5):
        start = time.perf_counter()
        c = a @ b
        cp.cuda.Device().synchronize()
        times.append(time.perf_counter() - start)
    print(f"{n} x {n} matrix multiply: {min(times) * 1000:.2f} ms")

## Part E, Step 3: Time the Same Test with PyTorch

In [ ]:
import time
import torch

device = "cuda"
sizes = [256, 512, 1024, 2048]

# Warm-up
w = torch.rand(64, 64, device=device)
_ = w @ w
torch.cuda.synchronize()

for n in sizes:
    a = torch.rand(n, n, device=device)
    b = torch.rand(n, n, device=device)
    torch.cuda.synchronize()

    times = []
    for _ in range(5):
        start = time.perf_counter()
        c = a @ b
        torch.cuda.synchronize()
        times.append(time.perf_counter() - start)
    print(f"{n} x {n} matrix multiply: {min(times) * 1000:.2f} ms")

## Part E, Step 4: Fill In the Comparison
Copy the CPU times from your Lab 4 table. The speed-up is the CPU time divided by the GPU time.

| Matrix size | CPU (ms) Lab 4 | CuPy (ms) | PyTorch (ms) | Speed-up (CPU / CuPy) |
|---|---|---|---|---|
| 256 x 256 | | | | |
| 512 x 512 | | | | |
| 1024 x 1024 | | | | |
| 2048 x 2048 | | | | |

> **Note.** The CPU result in Lab 4 was a single run with NumPy, and the GPU results here are the fastest of five runs. This favours the GPU a little. The difference is still large for big matrices, but read a small result such as the 256 size with care.

- At which size is the GPU speed-up the biggest?
- Why is the speed-up small for the smallest size?

## Challenge: What If You Forget to Synchronise?
Copy the CuPy timing cell into the cell below. Delete the line `cp.cuda.Device().synchronize()` inside the timing loop (right after `c = a @ b`). Run it. What happens to the times, and why can you not trust them?

In [ ]:
# Your challenge cell here

---
# Part F: The Jetson System Monitor (50 min)

This is the first project of the course. A monitor shows how healthy a computer is. On the Jetson everything you need is in the files of Linux (the folders `/proc` and `/sys`), so you need no extra program. The project reads the files, turns them into numbers, and shows a small dashboard.

| What | Where Linux keeps it | How we read it |
|---|---|---|
| Memory (shared by CPU and GPU) | `/proc/meminfo` | `MemTotal` and `MemAvailable` in kB |
| CPU load | `/proc/stat` | the time the CPU was busy, compared between two readings |
| GPU load | `/sys/devices/platform/gpu.0/load` | a number from 0 to 1000 (tenths of a percent) |
| Temperatures | `/sys/class/thermal/thermal_zone*/temp` | thousandths of a degree Celsius |

### Step 1: Read the Numbers
The cell has one small function for each reading. Each one returns a number, or `None` when the file cannot be read (a sensor can be busy for a moment). Run the cell.

In [ ]:
import csv
import glob
import os
import threading
import time
from datetime import datetime


def read_text(path):
    try:
        with open(path, "rb") as f:
            data = f.read()
        return data.decode(errors="replace").strip() if data is not None else None
    except (OSError, TypeError):          # a sensor can be busy for a moment
        return None


def memory():
    """(used MB, total MB, percent used). The memory is shared by the CPU and the GPU."""
    info = {}
    for line in read_text("/proc/meminfo").splitlines():
        key, value = line.split(":")
        info[key] = int(value.split()[0])            # kB
    total, available = info["MemTotal"], info["MemAvailable"]
    return (total - available) // 1024, total // 1024, 100 * (total - available) / total


_last_cpu = {"busy": 0, "total": 0}


def cpu_percent():
    """How busy the CPU was since the last call, in percent."""
    values = [int(v) for v in read_text("/proc/stat").splitlines()[0].split()[1:]]
    idle = values[3] + values[4]                     # idle + iowait
    total, busy = sum(values), sum(values) - idle
    d_total, d_busy = total - _last_cpu["total"], busy - _last_cpu["busy"]
    _last_cpu.update(busy=busy, total=total)
    return 100 * d_busy / d_total if d_total else 0.0


def gpu_percent():
    """GPU load in percent (the file holds tenths of a percent)."""
    value = read_text("/sys/devices/platform/gpu.0/load")
    return int(value) / 10 if value and value.isdigit() else None


def temperatures():
    """{sensor name: degrees Celsius} for the sensors that answer."""
    found = {}
    for zone in sorted(glob.glob("/sys/class/thermal/thermal_zone*")):
        name, value = read_text(zone + "/type"), read_text(zone + "/temp")
        if name and value and value.lstrip("-").isdigit() and name not in found:
            found[name] = int(value) / 1000
    return found


cpu_percent()                                        # the first call only sets the starting point
time.sleep(1)
used, total, percent = memory()
print(f"Memory : {used} / {total} MB ({percent:.0f} %)")
print(f"CPU    : {cpu_percent():.0f} %")
print(f"GPU    : {gpu_percent()} %")
print("Temperatures:", {k: round(v, 1) for k, v in temperatures().items()})

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Memory used and total:
- CPU and GPU load now:
- Which temperature sensors answered, and which is the highest:
- Why does cpu_percent need two calls (look at the code):

### Step 2: The Dashboard
`snapshot()` collects all readings with a timestamp. `bar(percent)` draws a bar of `#` characters. `dashboard(s)` makes the text of the screen. Run the cell: it prints one dashboard.

In [ ]:
def snapshot():
    used, total, percent = memory()
    return {"time": datetime.now(), "ram_used_mb": used, "ram_total_mb": total, "ram_percent": percent,
            "cpu_percent": cpu_percent(), "gpu_percent": gpu_percent(), "temperatures": temperatures()}


def bar(percent, width=30):
    filled = int(round(width * max(0.0, min(100.0, percent or 0.0)) / 100))
    return "[" + "#" * filled + "-" * (width - filled) + f"] {percent or 0:5.1f} %"


def dashboard(s):
    hottest = max(s["temperatures"].items(), key=lambda item: item[1]) if s["temperatures"] else ("?", 0.0)
    lines = [f" JETSON SYSTEM MONITOR            {s['time'].strftime('%Y-%m-%d %H:%M:%S')}",
             " " + "-" * 56,
             f" RAM  {bar(s['ram_percent'])}  {s['ram_used_mb']} / {s['ram_total_mb']} MB",
             f" CPU  {bar(s['cpu_percent'])}",
             f" GPU  {bar(s['gpu_percent'])}",
             f" TEMP hottest: {hottest[0]} {hottest[1]:.1f} C   " + "  ".join(f"{k} {v:.0f}" for k, v in list(s["temperatures"].items())[:4])]
    return "\n".join(lines)


print(dashboard(snapshot()))

### Step 3: Watch It Live and Log It
`monitor` prints a new dashboard every second for `seconds` seconds, clears the old one, and adds one line to `system_monitor.csv` each time. To stop early, press the **Stop** square in the toolbar (Interrupt the kernel).

In [ ]:
from IPython.display import clear_output


def monitor(seconds=10, log="system_monitor.csv"):
    first = not os.path.exists(log)
    with open(log, "a", newline="") as f:
        writer = csv.writer(f)
        if first:
            writer.writerow(["time", "ram_used_mb", "cpu_percent", "gpu_percent", "hottest_c"])
        try:
            for _ in range(seconds):
                s = snapshot()
                writer.writerow([s["time"].strftime("%Y-%m-%d %H:%M:%S"), s["ram_used_mb"], round(s["cpu_percent"], 1),
                                 s["gpu_percent"], round(max(s["temperatures"].values(), default=0.0), 1)])
                clear_output(wait=True)
                print(dashboard(s))
                time.sleep(1)
        except KeyboardInterrupt:
            print("\nStopped.")


monitor(8)

### Step 4: Give the GPU Some Work
The cell starts a thread that multiplies big matrices on the GPU for 10 seconds, and runs the monitor at the same time. Watch the **GPU** bar: it should rise from near 0 % to a high value, and the temperature creeps up. This is the same effect that you saw in `jtop`.

In [ ]:
import cupy as cp


def gpu_work(seconds=10):
    a = cp.random.rand(2048, 2048, dtype=cp.float32)
    end = time.time() + seconds
    while time.time() < end:
        a = a @ a
        a /= float(cp.abs(a).max()) + 1e-6           # keep the numbers small
        cp.cuda.Device().synchronize()


before = gpu_percent()
worker = threading.Thread(target=gpu_work, args=(10,))
worker.start()
peak = 0.0
for _ in range(8):
    time.sleep(1)
    peak = max(peak, gpu_percent() or 0.0)
monitor(3)
worker.join()
print(f"GPU load before: {before} % | highest load while working: {peak} %")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- GPU load before the work and the highest value during the work:
- Did the temperature change, and by how much:
- What did the CPU bar do while the GPU worked:

### Step 5: Make It Your Own
Add **at least one improvement**, or one of your own:

- Show a warning line `HOT` when the hottest sensor is above 70 degrees, and `LOW MEMORY` when less than 10 % of the memory is free.
- Add the power mode (`sudo nvpmodel -q` after `labview.sudo_login()`) to the dashboard.
- Show the six CPU cores one by one (read `cpu0` ... in `/proc/stat`).
- Draw the RAM history of the last 60 seconds as a line of `#` characters.
- Save a picture of the dashboard with `cv2.putText` on a black image (Lab 6).

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Improvement you chose:
- Which lines did you change:
- What did you see:

> **Project milestone 1: Jetson System Monitor.** Show the instructor a dashboard with the **memory, the CPU and GPU activity, the temperatures and a timestamp**, running live for 10 seconds, and the file `system_monitor.csv`. Explain where each number comes from (which file).

> **Where next.** The GPU is only useful if the data is already there. In Lab 6 you capture pictures from the camera and prepare them with OpenCV, and in Lab 7 the GPU runs an AI model on them.

## 4. Check Task
Show your instructor. The instructor initials each line when it is done correctly.

| No. | Task to demonstrate | Initials |
|---|---|---|
| 1 | CUDA, CuPy and PyTorch are checked and the GPU properties are read | |
| 2 | The CPU and GPU timings are shown for four array sizes and the copy cost is explained | |
| 3 | The matrix timing table is complete (Lab 4 CPU, CuPy, PyTorch) with the power mode written | |
| 4 | The synchronise challenge is done and explained | |
| 5 | The monitor shows memory, CPU load, GPU load, temperatures and a timestamp | |
| 6 | The GPU bar rises when the GPU works (Step 4), and the log file has lines | |
| 7 | At least one improvement works and you explain the code you changed | |

## 5. Assessment Questions
Answer in one or two sentences.

**Your answers**

- 1. Why is a GPU good at AI? Use the words 'many' and 'at the same time'.:
- 2. Why is the GPU slower than the CPU for a very small array?:
- 3. Why must you call `synchronize()` before you stop the timer?:
- 4. What are the three times in Part C Step 2? Which one can be bigger than the calculation?:
- 5. On the Jetson the CPU and the GPU share one memory. What does this mean for the memory bar of the monitor?:
- 6. Where does the monitor read the GPU load from? In which unit is the number?:
- 7. Why does `cpu_percent` compare two readings and not use one?:
- 8. The hottest sensor shows 85 degrees. Name two things you can do.:
- 9. Which power mode gave the faster matrix multiplication? Why is the mode written next to every timing?:

## 6. Troubleshooting

| Symptom | Likely cause | What to try |
|---|---|---|
| `ModuleNotFoundError: cupy` | The Lab 0 setup did not finish | Call your instructor. Do not run `pip install` in this lab. |
| `CUDA available: False` | The PyTorch of your environment is the CPU version | Stop and call your instructor |
| The GPU is slower than the CPU | The array is small, or you timed the copy | Use bigger arrays; look at Part C Step 3 |
| `cpu_percent` shows 0 or 100 all the time | Only one reading was made, or very short time between readings | Call it again after one second |
| `gpu_percent` is `None` | The Jetson has no `gpu.0/load` file in this version of JetPack | Ask your instructor. You can read the GPU load with `jtop` instead. |
| A temperature sensor is missing | The sensor was busy for a moment | Run the cell again; the code skips sensors that do not answer |
| `CUDA out of memory` | Other notebooks use the shared 8 GB | Shut down other kernels (Kernel, Shut Down All Kernels) |
| The dashboard does not stop | `monitor` runs for `seconds` seconds | Press the Stop square in the toolbar |
| The notebook looks broken | You changed or deleted too much | Close it and run `import labview; labview.reset(5)` in a cell of a new notebook (File, New, Notebook). You get a fresh copy. |

## 7. Further Reading
- CuPy: https://docs.cupy.dev/
- PyTorch CUDA semantics: https://pytorch.org/docs/stable/notes/cuda.html
- NVIDIA Jetson, power and performance (tegrastats, jtop): https://docs.nvidia.com/jetson/archives/r36.4/DeveloperGuide/SD/PlatformPowerAndPerformance.html